# 00 — Initial Dataset Cleanup (Group Leader)

This notebook documents the initial cleanup applied to the original scraped dataset
before distributing raw_dataset.csv to the preprocessing teams.
It is provided for full transparency and reproducibility of the pipeline.

**This notebook does not need to be re-run by team members.**

In [ ]:
import pandas as pd
import numpy as np

RAW_URL = "https://raw.githubusercontent.com/MamoMGD1/ISE302-DataMining-GroupProject/main/data/raw_dataset.csv"
df = pd.read_csv(RAW_URL)
print(f"Shape: {df.shape}")
df.head(3)

In [ ]:
# Drop columns with high missingness, near-zero variance, or that are identifiers/redundant aggregates
cols_to_drop = [
    'Şanzıman',            # ~63% missing values
    'Ağır Hasarlı',        # ~53% missing + ~11% 'Belirtilmemiş' → >64% unreliable
    'Yakıt Tüketimi',      # redundant aggregate; split into separate columns already
    'Motor ve Performans', # redundant aggregate; data in dedicated columns
    'Plaka Uyruğu',        # near-zero variance: ~99.4% Turkish
    'Kopyalandı',          # ~72% unique values → near-identifier
    '_URL',                # unique identifier per record
    '_Başlık',             # unique identifier per record
]
df.drop(columns=cols_to_drop, inplace=True)
print(f"Shape after dropping columns: {df.shape}")

In [ ]:
# Extract 'Aks Aralığı' from 'Boyut ve Kapasite' before dropping the column
df['Aks Aralığı'] = (
    df['Boyut ve Kapasite']
    .str.extract(r'Aks Aralığı (\d+) mm')[0]
    .astype(float)
)
df.drop(columns=['Boyut ve Kapasite'], inplace=True)
print(f"Null count for 'Aks Aralığı': {df['Aks Aralığı'].isnull().sum()}")

In [ ]:
# Summary of the cleaned dataset
print(f"Final shape: {df.shape}")
print("\nColumns:")
print(df.columns.tolist())
print("\nNull counts per column:")
print(df.isnull().sum())

In [ ]:
from google.colab import files
df.to_csv('raw_dataset.csv', index=False)
files.download('raw_dataset.csv')